# Exercise 2: Kalman Sensor Fusion

Consider longitudinal car motion with state $x = [p, v, a]$ where $p$ is the position, $v$ is the velocity, and $a$ is the acceleration, and the control is the jerk, $j$. The kinematics for this system are:
\begin{equation}
\dot{p} = v, \quad \dot{v} = a, \quad \dot{a} = j,
\end{equation}
which we can discretize exactly with sampling time $T$ as:
\begin{equation}
p_{t+1} = p_t + v_t T + \frac{1}{2}a_tT^2 + \frac{1}{6}jT^3, \quad v_{t+1} = v_t + a_t T + \frac{1}{2}jT^2, \quad a_{t+1} = a_t + j T.
\end{equation}
We also assume there are three sensors we could have on the vehicle:
1. An IMU that gives a measurement of acceleration $a$.
2. A GNSS sensor that gives a direct measurement of vehicle position, $p$.
3. A lidar sensor that gives a direct measurement of vehicle position, $p$, via range to a known object.

In [ ]:
import numpy as np
from scipy.linalg import block_diag
from utils import plot_car_estimation, rmse, observable
from typing import Optional

# Define process and measurement noises
Q = np.diag([0.0, 0.01, 0.0]) # process noise
std_imu = 0.25
std_gnss = 5.0
std_lidar = 5.0

In [ ]:
def simulate(
    x0: np.ndarray, 
    u: np.ndarray, 
    A: np.ndarray, 
    B: np.ndarray, 
    C: np.ndarray, 
    Q: np.ndarray, 
    R: np.ndarray, 
    mean0: np.ndarray, 
    cov0: np.ndarray, 
    num_sims: int, 
    bias: Optional[np.ndarray]=None,
) -> tuple[np.ndarray, np.ndarray, float]:
    """
    Simulate the linear system and KF estimation. Compute RSME
    to estimate error of the filter across time and over a batch of runs.

    Args:
        x0: initial state of the system, shape (n,)
        u: controls to apply at each time step, shape (m, N)
        A: dynamics matrix, shape (n, n)
        B: control matrix, shape (n, m)
        C: measurement matrix, shape (o, n)
        Q: process noise covariance, shape (n, n)
        R: measurement noise covariance, shape (o, o)
        mean0: initial KF mean, shape (n,)
        cov0: initial KF covariance, shape (n, n)
        num_sims: number of simulations to batch for RMSE calc
        bias: bias to the measurement, or None to not add a bias

    Returns:
        Actual state trajectory from the last batch run, shape (n, N + 1).
        KF mean estimate trajectory from the last batch, shape (n, N + 1).
        Average RMSE over the batch of runs.
    """
    total_pos_rmse = 0
    for run in range(num_sims):
        n = x0.size
        o = C.shape[0]
        N = u.shape[1]
        x = np.zeros((n, N + 1))
        x[:, 0] = x0
        mean = np.zeros((n, N + 1))
        mean[:, 0] = mean0
        cov = cov0
        rng = np.random.default_rng(run) # for determinism
        for i in range(N):
            # Simulate dynamics
            ϵ = rng.multivariate_normal(np.zeros((Q.shape[0],)), Q)
            x[:, i + 1] = A @ x[:, i] + B @ u[:, i] + ϵ
        
            # Simulate measurement
            δ = rng.multivariate_normal(np.zeros((R.shape[0],)), R)
            b = np.zeros(o) if bias is None else bias
            z = C @ x[:, i] + b + δ
        
            # Estimation
            mean[:, i + 1], cov = kalman_filter_update(A, B, C, Q, R, mean[:, i], cov, u[:, i], z)

        # Position error metric
        total_pos_rmse += rmse(x[0, :], mean[0, :])
    return x, mean, total_pos_rmse / num_sims

### Exercise 2.1 Implement Kalman Filter
First, define matrices $A$ and $B$ for the car's dynamics model described above in discrete time. Use the variable `T` as the sampling time. Then, implement the function `kalman_filter_update` to perform a single step of a Kalman filter update.

In [ ]:
T = 0.01 # sampling time
##### YOUR CODE STARTS HERE #####
# Longitudinal car dynamics model (discrete time)
raise NotImplementedError("Need to implement code here.")
###### YOUR CODE ENDS HERE ######

In [ ]:
def kalman_filter_update(
    A: np.ndarray, 
    B: np.ndarray, 
    C: np.ndarray, 
    Q: np.ndarray, 
    R: np.ndarray, 
    mean: np.ndarray, 
    cov: np.ndarray, 
    u: np.ndarray, 
    z: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Kalman filter update step.

    Args:
        A: dynamics matrix, shape (n, n)
        B: control matrix, shape (n, m)
        C: measurement matrix, shape (o, n)
        Q: process noise covariance, shape (n, n)
        R: measurement noise covariance, shape (o, o)
        mean: mean estimate, shape (n,)
        cov: estimate covariance, shape (n, n)
        u: control input, shape (m,)
        z: measurement, shape (o,)

    Returns:
        Posterior mean and covariance.
    """
    ##### YOUR CODE STARTS HERE #####
    # NOTE: Implement Kalman Filtering Predict and Update steps.
    # Write resulting means to mean.
    # Write resulting covariances to cov.
    raise NotImplementedError("Need to implement code here.")
    ###### YOUR CODE ENDS HERE ######
    return mean, cov

### Exercise 2.2: KF Sensor Fusion for Autonomous Car
First, run the code below to simulate the system with a provided open-loop control input and see what the estimates look like with no sensors to correct the estimate. We will compute the average RMSE error of the position estimate, and us this to compare across the different solutions later.

In [ ]:
# Form an interesting jerk input sequence
u = np.hstack([np.ones(100), -1*np.ones(150), np.ones(50), np.zeros(100), -np.ones(75), np.ones(100), -0.25*np.ones(100), np.zeros(100)])
u = u.reshape((1, u.size))

# Assume we don't know the exact initial condition
x0 = np.array([10, 5, 0.1])

# First run a case where we have no measurements to see the error
C = np.array([[0, 0, 0]])
R = np.diag([1])

# Simulate
num_sims = 10
mean0 = np.zeros(3)
cov0 = np.diag([100.0, 10.0, 1.0])
x, mean, p_error_none = simulate(x0, u, A, B, C, Q, R, mean0, cov0, num_sims)
print(f"State is observable with sensor config: {observable(A, C)}")

# Compute RMSE for position
print(f"Position RMSE: {p_error_none:.3f}")

# Plot to see accuracy
plot_car_estimation(x, mean, T)

Now, implement the $C$ and $R$ matrices below to define a setup that has only an IMU sensor. Run the code to see how the position error estimate is impacted. Is this what you would expect? Use the variable `std_imu` for the IMU sensor noise standard deviation.

In [ ]:
##### YOUR CODE STARTS HERE #####
# Define measurement model for IMU only
raise NotImplementedError("Need to implement code here.")
###### YOUR CODE ENDS HERE ######

# Simulate
x, mean, p_error_imu = simulate(x0, u, A, B, C, Q, R, mean0, cov0, num_sims)
print(f"State is observable with sensor config: {observable(A, C)}")

# Compute RMSE for position
improvement = 100*(p_error_imu - p_error_none) / p_error_none
print(f"IMU Position RMSE: {p_error_imu:.3f}")
print(f"RMSE change compared to no sensors: {improvement:.2f}%")

# Plot to see accuracy
plot_car_estimation(x, mean, T)

Repeat the above exercise but now define $C$ and $R$ with an IMU + lidar sensor setup. Use the variables `std_imu` and `std_lidar` for the IMU sensor noise standard deviation. The error should significantly improve!

In [ ]:
##### YOUR CODE STARTS HERE #####
# Define measurement model for IMU + lidar
raise NotImplementedError("Need to implement code here.")
###### YOUR CODE ENDS HERE ######

# Simulate
x, mean, p_error_imu_lidar = simulate(x0, u, A, B, C, Q, R, mean0, cov0, num_sims)
print(f"State is observable with sensor config: {observable(A, C)}")

# Compute RMSE for position
improvement = 100*(p_error_imu_lidar - p_error_none) / p_error_none
print(f"IMU + lidar Position RMSE: {p_error_imu_lidar:.3f}")
print(f"RMSE change compared to no sensors: {improvement:.2f}%")

# Plot to see accuracy
plot_car_estimation(x, mean, T)

Repeat the above exercise but now define $C$ and $R$ with an IMU + lidar + GNSS sensor setup. Use the variables `std_imu`, `std_lidar`, `std_gnss` for the IMU sensor noise standard deviation. Note that here we assume the GNSS sensor has a constant bias error. How do you think this will effect the position estimate?

In [ ]:
##### YOUR CODE STARTS HERE #####
# Define measurement model for IMU + lidar + GNSS with bias
raise NotImplementedError("Need to implement code here.")
###### YOUR CODE ENDS HERE ######

# GNSS bias
gnss_bias = 10.0
bias = np.array([0, 0, gnss_bias]) 

# Simulate
x, mean, p_error_imu_lidar_gnss = simulate(x0, u, A, B, C, Q, R, mean0, cov0, num_sims, bias)
print(f"State is observable with sensor config: {observable(A, C)}")

# Compute RMSE for position
improvement = 100*(p_error_imu_lidar_gnss - p_error_none) / p_error_none
improvement_lidar_only = 100*(p_error_imu_lidar_gnss - p_error_imu_lidar) / p_error_imu_lidar
print(f"IMU + lidar + GNSS Position RMSE: {p_error_imu_lidar_gnss:.3f}")
print(f"RMSE change compared to no sensors: {improvement:.2f}%")
print(f"RMSE change compared to IMU + lidar: {improvement_lidar_only:.2f}%")

# Plot to see accuracy
plot_car_estimation(x, mean, T)

### Exercise 2.3 Augmented State
Using the GNSS sensor above with bias has an impact on the ability to accurately estimate the position. In this part of the exercise, implement an augmented state version of the Kalman filter to estimate the GNSS bias. Specifically, define the variables `A_aug`, `B_aug`, `C_aug`, and `R_aug`.

In [ ]:
##### YOUR CODE STARTS HERE #####
# Define augmented state system
raise NotImplementedError("Need to implement code here.")
###### YOUR CODE ENDS HERE ######

# Simulate
x0_aug = np.hstack((x0, gnss_bias))
mean0_aug = np.hstack((mean0, 0))
cov0_aug = block_diag(cov0, 10)
x, mean, p_error_imu_lidar_aug_gnss = simulate(x0_aug, u, A_aug, B_aug, C_aug, Q_aug, R, mean0_aug, cov0_aug, num_sims, bias)
print(f"State + bias is observable with sensor config: {observable(A_aug, C_aug)}")

# Compute RMSE for position
improvement = 100*(p_error_imu_lidar_aug_gnss - p_error_none) / p_error_none
improvement_lidar_only = 100*(p_error_imu_lidar_aug_gnss - p_error_imu_lidar) / p_error_imu_lidar
improvement_lidar_bias_gnss = 100*(p_error_imu_lidar_aug_gnss - p_error_imu_lidar_gnss) / p_error_imu_lidar_gnss
print(f"IMU + lidar + augmented GNSS Position RMSE: {p_error_imu_lidar_aug_gnss:.3f}")
print(f"RMSE change compared to no sensors: {improvement:.2f}%")
print(f"RMSE change compared to IMU + lidar: {improvement_lidar_only:.2f}%")
print(f"RMSE change compared to IMU + lidar + (non-augmented) GNSS: {improvement_lidar_bias_gnss:.2f}%")

# Plot to see accuracy
plot_car_estimation(x, mean, T)